# Circuit Translation Workflow

This notebook demonstrates SDK-facing circuit translation workflows. It keeps the workflow compact while still showing preflight inspection, exact semantic verification, saved reports, runnable output generation, and unsupported-construct diagnostics.

Use this when you have a small static circuit written for one free local SDK and want to move the circuit construction to another supported SDK while preserving measurement behavior.

## Problem

A circuit may be written in one SDK even though the same gate sequence is portable across local simulators. This workflow translates supported static circuit snippets through the package's neutral circuit model, then verifies the translated circuit before it is used downstream.

## Translation Scope

The workflow is intentionally conservative. It supports static circuit construction, common one- and two-qubit gates, static numeric parameters, simple `range(...)` loops, OpenQASM, internal JSON, and local Python SDK circuit APIs. It rejects dynamic program behavior instead of guessing.

## Variables and Parameters

- `from_format`: source circuit format, such as `qiskit`, `cirq`, `pennylane`, `braket`, `openqasm`, or `internal-json`.
- `to_format`: output format, such as `cirq`, `qiskit_aer`, `pennylane`, `braket_local`, `openqasm`, or `internal-json`.
- `verify`: semantic verification mode, here `exact`.
- `ARTIFACT_DIR`: notebook artifact directory for translated source and report JSON.

## Setup

In [ ]:
import json

import pandas as pd
from IPython.display import display

from quantum_backend_bench.core.circuit_translate import (
    TranslationError,
    import_circuit_source,
    translate_circuit_source,
    translation_check_report,
    translation_error_report,
    translation_result_report,
)
from quantum_backend_bench.core.draw import draw_benchmark
from quantum_backend_bench.utils.notebook import notebook_artifact_dir, verification_frame

ARTIFACT_DIR = notebook_artifact_dir()
MIGRATION_TARGET = "cirq"
TRANSLATION_TARGETS = [
    {"format": "qiskit_aer", "label": "Qiskit Aer", "extension": ".py"},
    {"format": "cirq", "label": "Cirq", "extension": ".py"},
    {"format": "pennylane", "label": "PennyLane", "extension": ".py"},
    {"format": "braket_local", "label": "Braket LocalSimulator", "extension": ".py"},
]
DIAGRAM_BACKENDS = [target["format"] for target in TRANSLATION_TARGETS]


def source_preview(title, source, max_lines=28):
    lines = source.strip().splitlines()
    shown = lines[:max_lines]
    print(title)
    print("-" * len(title))
    for number, line in enumerate(shown, start=1):
        print(f"{number:>2}: {line}")
    if len(lines) > max_lines:
        print(f"... {len(lines) - max_lines} more lines")


def preflight_frame(report):
    contract = report["semantic_contract"]
    audit = report["migration_audit"]
    return pd.DataFrame(
        [
            {"field": "input format", "value": report["input_format"]},
            {"field": "target", "value": audit["target"]},
            {"field": "guarantee", "value": contract["guarantee"]},
            {"field": "qubits", "value": report["n_qubits"]},
            {"field": "operations", "value": report["operation_count"]},
            {
                "field": "measurements",
                "value": ", ".join(str(measurement) for measurement in report["measurements"]),
            },
            {
                "field": "gate counts",
                "value": ", ".join(
                    f"{gate}: {count}" for gate, count in report["gate_counts"].items()
                ),
            },
        ]
    )


def migration_audit_frame(report):
    audit = report["migration_audit"]
    rows = []
    for field in ("preserved", "rewritten", "rejected_if_present", "not_modeled"):
        for item in audit[field]:
            rows.append({"category": field, "detail": item})
    rows.append({"category": "verification", "detail": audit["verification_recommendation"]})
    return pd.DataFrame(rows)


def diagnostics_frame(report):
    rows = report.get("diagnostics", [])
    if not rows:
        return pd.DataFrame([{"severity": "none", "code": "-", "message": "No diagnostics."}])
    return pd.DataFrame(rows)


def translation_summary_frame(results):
    rows = []
    for target in TRANSLATION_TARGETS:
        target_format = target["format"]
        result = results[target_format]
        verification = result.verification
        rows.append(
            {
                "target": target["label"],
                "format": target_format,
                "verified": verification.passed if verification else None,
                "total_variation_distance": (
                    verification.total_variation_distance if verification else None
                ),
                "diagnostics": len(result.diagnostics),
                "notes": "; ".join(result.notes),
                "source_lines": len(result.source.strip().splitlines()),
            }
        )
    return pd.DataFrame(rows)


def verification_summary_frame(label, verification):
    return pd.DataFrame(
        [
            {
                "translation": label,
                "mode": verification.mode,
                "passed": verification.passed,
                "total_variation_distance": verification.total_variation_distance,
                "tolerance": verification.tolerance,
                "details": verification.details,
            }
        ]
    )


def artifact_frame(rows):
    return pd.DataFrame(rows, columns=["artifact", "target", "path"])


def draw_comparison(benchmark, backends):
    rows = []
    for backend in backends:
        try:
            diagram = draw_benchmark(benchmark, backend)
        except (
            Exception
        ) as exc:  # Optional SDK extras may not be installed in every notebook environment.
            rows.append(
                {
                    "backend": backend,
                    "status": "not available",
                    "diagram": f"{type(exc).__name__}: {exc}",
                }
            )
            continue
        rows.append({"backend": backend, "status": "drawn", "diagram": diagram})
    return rows


def display_diagram_comparison(rows):
    display(pd.DataFrame([{"backend": row["backend"], "status": row["status"]} for row in rows]))
    for row in rows:
        title = f"{row['backend']} native diagram ({row['status']})"
        print()
        source_preview(title, row["diagram"], max_lines=36)

## Source Circuit

Start with a small Qiskit circuit snippet that uses registers and a static rotation angle. The translator parses the circuit construction without executing SDK code.

In [ ]:
qiskit_source = """
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
import math

q = QuantumRegister(2, "q")
c = ClassicalRegister(2, "c")
theta = math.pi / 2

circuit = QuantumCircuit(q, c)
circuit.h(q[0])
circuit.rx(theta, q[1])
circuit.cx(q[0], q[1])
circuit.measure(q[0], c[0])
circuit.measure(q[1], c[1])
"""

source_preview("Input Qiskit source", qiskit_source)

## Preflight Inspection

`import_circuit_source` and `translation_check_report` provide the same basic information as `quantum-bench translate-check`: detected format, qubit count, gate inventory, measurements, diagnostics, and supported output formats. Passing `to_format` adds a target-aware `migration_audit` and the explicit `semantic_contract` that defines what is preserved, rewritten, rejected, and left outside the model.


In [ ]:
benchmark, detected_format = import_circuit_source(qiskit_source, from_format="qiskit")
check_report = translation_check_report(
    benchmark,
    detected_format,
    source_path="inline:qiskit_source",
    to_format=MIGRATION_TARGET,
)

display(preflight_frame(check_report))
display(migration_audit_frame(check_report))
display(diagnostics_frame(check_report))

## Visual Circuit Comparison

Draw the same neutral circuit through each local SDK backend. These diagrams are a structural sanity check for a person reading the notebook; exact semantic verification comes from the probability comparisons below.

In [ ]:
diagram_rows = draw_comparison(benchmark, DIAGRAM_BACKENDS)
display_diagram_comparison(diagram_rows)

## Translate to All Local SDK Targets

Translate the same source circuit to every currently supported free/local SDK output. Each target is verified by comparing neutral exact probabilities.

In [ ]:
translation_results = {}
for target in TRANSLATION_TARGETS:
    translation_results[target["format"]] = translate_circuit_source(
        qiskit_source,
        from_format="qiskit",
        to_format=target["format"],
        verify="exact",
    )

display(translation_summary_frame(translation_results))
for target in TRANSLATION_TARGETS:
    result = translation_results[target["format"]]
    display(verification_summary_frame(target["label"], result.verification))

## Save Translation Artifacts

Save one translated source file per target plus a combined JSON report that records verification, notes, and diagnostics.

In [ ]:
artifact_rows = []
combined_report = []
for target in TRANSLATION_TARGETS:
    target_format = target["format"]
    result = translation_results[target_format]
    source_path = ARTIFACT_DIR / f"translation_qiskit_to_{target_format}{target['extension']}"
    report = translation_result_report(
        result,
        source_path="inline:qiskit_source",
        from_format="qiskit",
        to_format=target_format,
    )
    source_path.write_text(result.source, encoding="utf-8")
    combined_report.append(report)
    artifact_rows.append(
        {"artifact": "translated source", "target": target_format, "path": str(source_path)}
    )

combined_report_path = ARTIFACT_DIR / "translation_qiskit_all_targets_report.json"
combined_report_path.write_text(
    json.dumps(combined_report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
artifact_rows.append(
    {"artifact": "combined report", "target": "all", "path": str(combined_report_path)}
)

artifact_frame(artifact_rows)

## Translated Source Previews

Inspect each generated SDK source file in the notebook without opening the saved artifacts separately.

In [ ]:
for target in TRANSLATION_TARGETS:
    result = translation_results[target["format"]]
    source_preview(f"{target['label']} translated source", result.source, max_lines=42)
    print()

## Runnable Script Output

The same translation path can emit a runnable script. This example uses the Cirq target because it is lightweight and fully local.

In [ ]:
runner_result = translate_circuit_source(
    qiskit_source,
    from_format="qiskit",
    to_format="cirq",
    include_runner=True,
    runner_shots=128,
    verify="exact",
)
runner_path = ARTIFACT_DIR / "translation_qiskit_to_cirq_runner.py"
runner_path.write_text(runner_result.source, encoding="utf-8")

source_preview("Runnable Cirq script preview", runner_result.source, max_lines=40)
display(
    artifact_frame([{"artifact": "runnable script", "target": "cirq", "path": str(runner_path)}])
)

## Unsupported Construct Diagnostics

Unsupported dynamic behavior fails with stable diagnostic codes. This example uses a runtime value for a rotation parameter, so translation stops instead of guessing.

In [ ]:
unsupported_source = """
from qiskit import QuantumCircuit

theta = get_theta()
circuit = QuantumCircuit(1)
circuit.rx(theta, 0)
"""

try:
    import_circuit_source(unsupported_source, from_format="qiskit")
except TranslationError as exc:
    error_report = translation_error_report(
        exc,
        source_path="inline:unsupported_source",
        from_format="qiskit",
    )
    display(diagnostics_frame(error_report))

## Verification Summary

In [ ]:
checks = []
for target in TRANSLATION_TARGETS:
    verification = translation_results[target["format"]].verification
    checks.append(
        {
            "check": f"{target['label']} exact verification",
            "value": verification.total_variation_distance,
            "expected": "<= 1e-09",
            "passed": verification.passed,
        }
    )
checks.append(
    {
        "check": "Runnable Cirq output exact verification",
        "value": runner_result.verification.total_variation_distance,
        "expected": "<= 1e-09",
        "passed": runner_result.verification.passed,
    }
)

verification_frame(checks)